# StrikeVision — Pose Estimation con merge de fragmentos

Variante del pipeline de pose que resuelve la **fragmentación de ByteTrack** antes de estimar keypoints. En vez de exigir que el detector entregue exactamente dos tracks, los fragmentos se agrupan en **dos identidades estables** (`fighter_left` / `fighter_right`) por continuidad temporal y espacial.

In [1]:
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
import yaml

from ufc_tracker.detection.personDetection import track_video
from ufc_tracker.detection.weights import project_root
from ufc_tracker.pose.estimator import MediaPipePoseEstimator
from ufc_tracker.pose.pipeline import (
    calculate_metrics,
    estimate_pose_records,
    render_pose_preview,
)
from ufc_tracker.tracking.contracts import TrackingRecord

ROOT = project_root(Path.cwd())
CONFIG_PATH = ROOT / 'configs/app/pose_pipeline.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config

{'output_root': 'data/processed/poses',
 'tracking_confidence': 0.5,
 'min_track_frames': 15,
 'max_frames': None}

## Seleccionar un round

La tabla usa el manifiesto versionado del dataset. Cambia el índice de `manifest.iloc[...]` por cualquier fila antes de ejecutar el pipeline.

In [2]:
manifest = pd.read_csv(ROOT / 'data/metadata/splits_manifest.csv')
display(manifest[['category', 'fight_id', 'round_number', 'relative_path']])
VIDEO_RELATIVE_PATH = manifest.iloc[33]['relative_path']
VIDEO_PATH = ROOT / VIDEO_RELATIVE_PATH
OUTPUT_DIR = ROOT / 'outputs/pose-merge' / VIDEO_PATH.stem
MAX_FRAMES = 1000
VIDEO_PATH, OUTPUT_DIR

,category,fight_id,round_number,relative_path
0,aggressive_men,adesanya_pereira_1,1,data/splits/aggressive_men/adesanya_pereira_1_...
1,aggressive_men,adesanya_pereira_1,2,data/splits/aggressive_men/adesanya_pereira_1_...
2,aggressive_men,adesanya_pereira_1,3,data/splits/aggressive_men/adesanya_pereira_1_...
3,aggressive_men,adesanya_pereira_1,4,data/splits/aggressive_men/adesanya_pereira_1_...
4,aggressive_men,adesanya_pereira_1,5,data/splits/aggressive_men/adesanya_pereira_1_...
5,aggressive_men,holloway_gaethje,1,data/splits/aggressive_men/holloway_gaethje__h...
6,aggressive_men,holloway_gaethje,2,data/splits/aggressive_men/holloway_gaethje__h...
7,aggressive_men,holloway_gaethje,3,data/splits/aggressive_men/holloway_gaethje__h...
8,aggressive_men,holloway_gaethje,4,data/splits/aggressive_men/holloway_gaethje__h...
9,aggressive_men,holloway_gaethje,5,data/splits/aggressive_men/holloway_gaethje__h...


(WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/data/splits/normal_men/fiziev_bahamondes__rafael_fiziev_vs_ignacio_bahamondes__normal_men_round1.mp4'),
 WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/pose-merge/fiziev_bahamondes__rafael_fiziev_vs_ignacio_bahamondes__normal_men_round1'))

## Fase 1: tracking y descripción de fragmentos

`track_video()` devuelve una observación por frame y `track_id`. En un round completo ByteTrack parte al mismo peleador en varios fragmentos (cambios de cámara, oclusiones, clinch), así que antes de decidir quién es quién se describe cada fragmento: rango de frames, centroides de entrada y salida, tamaño y apariencia.

In [3]:
TRACKING_CONFIDENCE = 0.5

per_frame, stats, N_FRAMES = track_video(
    VIDEO_PATH, max_frames=MAX_FRAMES, conf=TRACKING_CONFIDENCE
)

capture = cv2.VideoCapture(str(VIDEO_PATH))
FPS = float(capture.get(cv2.CAP_PROP_FPS))
capture.release()

print(f'frames={N_FRAMES} fps={FPS} fragmentos={len(stats)}')

frames=1000 fps=30.0 fragmentos=73


In [4]:
# Umbrales de apariencia heredados de person detection
FIGHTER_SKIN_MIN = 0.50
FIGHTER_MIN_AREA = 0.01
MIN_FRAGMENT_FRAMES = 5


def describe_fragments(per_frame, stats):
    """Resume cada track_id: rango temporal, centroides, tamaño y apariencia."""
    tracks = {}
    for frame_index, frame_map in enumerate(per_frame):
        for tid, obs in frame_map.items():
            x1, y1, x2, y2 = obs.bbox_xyxy
            entry = tracks.setdefault(
                tid, {'frames': [], 'centroids': [], 'diagonals': []}
            )
            entry['frames'].append(frame_index)
            entry['centroids'].append(((x1 + x2) * 0.5, (y1 + y2) * 0.5))
            entry['diagonals'].append(float(np.hypot(x2 - x1, y2 - y1)))

    fragments = {}
    for tid, entry in tracks.items():
        count = stats[tid]['n']
        fragments[tid] = {
            'track_id': tid,
            'count': count,
            'frames': set(entry['frames']),
            'first_frame': entry['frames'][0],
            'last_frame': entry['frames'][-1],
            'first_centroid': entry['centroids'][0],
            'last_centroid': entry['centroids'][-1],
            'diagonal': float(np.median(entry['diagonals'])),
            'mean_cx': float(np.mean([c[0] for c in entry['centroids']])),
            'skin': stats[tid]['skin'] / count,
            'area': stats[tid]['area'] / count,
        }
    return fragments


def select_candidates(fragments):
    """Deja solo fragmentos con torso desnudo, area suficiente y algo de duración."""
    return {
        tid: f
        for tid, f in fragments.items()
        if f['skin'] >= FIGHTER_SKIN_MIN
        and f['area'] >= FIGHTER_MIN_AREA
        and f['count'] >= MIN_FRAGMENT_FRAMES
    }


fragments = describe_fragments(per_frame, stats)
candidates = select_candidates(fragments)

print(f'candidatos: {len(candidates)} / {len(fragments)}\n')
for tid in sorted(candidates, key=lambda t: candidates[t]['first_frame']):
    f = candidates[tid]
    print(
        f"  tid={tid:>4} n={f['count']:>4} "
        f"frames=[{f['first_frame']:>4}..{f['last_frame']:>4}] "
        f"skin={f['skin']:.2f} area={f['area']:.4f} "
        f"start={tuple(round(v) for v in f['first_centroid'])} "
        f"end={tuple(round(v) for v in f['last_centroid'])} "
        f"diag={f['diagonal']:.0f}"
    )

candidatos: 9 / 73

  tid=   1 n=  29 frames=[   0..  28] skin=0.85 area=0.0592 start=(1427, 505) end=(1060, 525) diag=561
  tid=   2 n= 582 frames=[   0.. 589] skin=0.82 area=0.1697 start=(457, 544) end=(638, 608) diag=933
  tid=  61 n= 846 frames=[  36.. 957] skin=0.81 area=0.1630 start=(1037, 270) end=(801, 435) diag=934
  tid= 113 n=  13 frames=[  72..  91] skin=0.84 area=0.0406 start=(1052, 261) end=(1102, 248) diag=408
  tid= 636 n=  10 frames=[ 496.. 522] skin=0.66 area=0.0141 start=(1652, 550) end=(1751, 573) diag=240
  tid= 687 n=  49 frames=[ 532.. 589] skin=0.63 area=0.0214 start=(1557, 602) end=(1604, 662) diag=289
  tid= 807 n= 407 frames=[ 591.. 999] skin=0.89 area=0.1628 start=(870, 432) end=(1073, 535) diag=948
  tid=1540 n=   6 frames=[ 974.. 979] skin=0.97 area=0.0239 start=(1105, 221) end=(1114, 202) diag=321
  tid=1553 n=   9 frames=[ 977.. 999] skin=0.79 area=0.0572 start=(1110, 398) end=(1290, 441) diag=620


## Fase 2: merge de fragmentos

Dos fragmentos son el **mismo peleador** cuando cumplen tres condiciones a la vez:

1. **No se solapan en el tiempo** — si aparecen en el mismo frame son personas distintas.
2. **El hueco es corto** — como máximo `MAX_MERGE_GAP_FRAMES` entre el final de uno y el inicio del otro.
3. **Están cerca en pantalla** — distancia entre centroides normalizada por la diagonal de la caja, por debajo de `MAX_MERGE_DISTANCE`.

El recorrido es codicioso en orden de aparición: cada fragmento se engancha al slot compatible más cercano, o abre uno nuevo. Al final se conservan los **dos slots con más frames**, ordenados de izquierda a derecha para mantener la convención de color azul/rojo.

La regla de no solape garantiza la invariante que el pipeline original exigía por la fuerza: dentro de un slot nunca hay dos cajas en el mismo frame, así que con dos slots hay **como máximo dos peleadores por frame**.

In [5]:
MAX_MERGE_GAP_FRAMES = 60  # 2 s a 30 fps
MAX_MERGE_DISTANCE = 0.5   # fracción de la diagonal de la caja
FIGHTER_LABELS = ('fighter_left', 'fighter_right')


def normalized_distance(slot, fragment):
    """Distancia entre el ultimo centroide del slot y el primero del fragmento."""
    ax, ay = slot['last_centroid']
    bx, by = fragment['first_centroid']
    scale = max(1.0, (slot['diagonal'] + fragment['diagonal']) * 0.5)
    return float(np.hypot(bx - ax, by - ay)) / scale


def merge_fragments(candidates):
    """Agrupa fragmentos en slots por continuidad temporal y espacial."""
    slots = []
    for tid in sorted(candidates, key=lambda t: candidates[t]['first_frame']):
        fragment = candidates[tid]
        target = None
        for slot in slots:
            if slot['frames'] & fragment['frames']:
                continue
            gap = fragment['first_frame'] - slot['last_frame']
            if gap < 0 or gap > MAX_MERGE_GAP_FRAMES:
                continue
            distance = normalized_distance(slot, fragment)
            if distance > MAX_MERGE_DISTANCE:
                continue
            if target is None or distance < target[1]:
                target = (slot, distance)

        if target is None:
            slots.append(
                {
                    'track_ids': [tid],
                    'frames': set(fragment['frames']),
                    'last_frame': fragment['last_frame'],
                    'last_centroid': fragment['last_centroid'],
                    'diagonal': fragment['diagonal'],
                    'mean_cx': fragment['mean_cx'],
                }
            )
            continue

        slot = target[0]
        slot['track_ids'].append(tid)
        slot['frames'] |= fragment['frames']
        slot['last_frame'] = fragment['last_frame']
        slot['last_centroid'] = fragment['last_centroid']
        slot['diagonal'] = fragment['diagonal']
        slot['mean_cx'] = float(np.mean([slot['mean_cx'], fragment['mean_cx']]))
    return slots


def select_fighter_slots(slots):
    """Top-2 slots por frames, ordenados de izquierda a derecha."""
    fighters = sorted(slots, key=lambda s: -len(s['frames']))[:2]
    fighters.sort(key=lambda s: s['mean_cx'])
    return fighters

In [6]:
slots = merge_fragments(candidates)
FIGHTERS = select_fighter_slots(slots)

print(f'slots={len(slots)}')
for slot in sorted(slots, key=lambda s: -len(s['frames'])):
    print(
        f"  ids={slot['track_ids']} frames={len(slot['frames'])} "
        f"mean_cx={slot['mean_cx']:.0f}"
    )

print()
for label, slot in zip(FIGHTER_LABELS, FIGHTERS):
    print(f"{label}: ids={slot['track_ids']} frames={len(slot['frames'])}")

crowded = [
    i
    for i, fm in enumerate(per_frame)
    if sum(any(t in fm for t in s['track_ids']) for s in FIGHTERS) > 2
]
covered = len(FIGHTERS[0]['frames'] | FIGHTERS[1]['frames']) / N_FRAMES
print(f'\nframes con mas de dos peleadores: {len(crowded)}')
print(f'cobertura con al menos un peleador: {covered:.3f}')

slots=6
  ids=[2, 807] frames=989 mean_cx=975
  ids=[1, 61, 1553] frames=884 mean_cx=1196
  ids=[687] frames=49 mean_cx=1430
  ids=[113] frames=13 mean_cx=1049
  ids=[636] frames=10 mean_cx=1728
  ids=[1540] frames=6 mean_cx=1108

fighter_left: ids=[2, 807] frames=989
fighter_right: ids=[1, 61, 1553] frames=884

frames con mas de dos peleadores: 0
cobertura con al menos un peleador: 1.000


## Fase 3: pose sobre identidades fusionadas

Con los slots resueltos se construyen los `TrackingRecord` directamente, sin pasar por `extract_fighter_tracking()`. Dos diferencias frente al pipeline original:

- El `fighter_id` es estable (`fighter_left` / `fighter_right`) y el `track_id` conserva el fragmento de ByteTrack que aportó la caja, para poder auditar el merge.
- Se escriben también los frames **no visibles** (`visible=False`, `bbox=None`), de modo que el eje temporal quede completo y `mark_missing()` corte el suavizado de MediaPipe.

In [7]:
def build_tracking_records(per_frame, fighters, fps):
    """Un registro por frame y peleador, incluyendo los frames sin caja."""
    records = []
    for frame_index, frame_map in enumerate(per_frame):
        timestamp = frame_index / fps
        for label, slot in zip(FIGHTER_LABELS, fighters):
            visible_id = next(
                (tid for tid in slot['track_ids'] if tid in frame_map), None
            )
            if visible_id is None:
                records.append(
                    TrackingRecord(
                        frame_index=frame_index,
                        timestamp_seconds=timestamp,
                        fighter_id=label,
                        track_id=-1,
                        bbox_xyxy=None,
                        confidence=None,
                        visible=False,
                    )
                )
                continue
            observation = frame_map[visible_id]
            records.append(
                TrackingRecord(
                    frame_index=frame_index,
                    timestamp_seconds=timestamp,
                    fighter_id=label,
                    track_id=visible_id,
                    bbox_xyxy=tuple(float(v) for v in observation.bbox_xyxy.tolist()),
                    confidence=float(observation.confidence),
                    visible=True,
                )
            )
    return records


def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8', newline='\n') as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + '\n')

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tracking_records = build_tracking_records(per_frame, FIGHTERS, FPS)
print(f'registros={len(tracking_records)} visibles={sum(r.visible for r in tracking_records)}')

pose_records = estimate_pose_records(
    VIDEO_PATH, tracking_records, MediaPipePoseEstimator(), frame_count=N_FRAMES
)

write_jsonl(OUTPUT_DIR / 'tracking.jsonl', (r.to_dict() for r in tracking_records))
write_jsonl(OUTPUT_DIR / 'pose.jsonl', (r.to_dict() for r in pose_records))

PREVIEW_PATH = OUTPUT_DIR / 'pose_preview.mp4'
render_pose_preview(VIDEO_PATH, pose_records, PREVIEW_PATH, frame_count=N_FRAMES)

metrics = calculate_metrics(pose_records)
(OUTPUT_DIR / 'pose_metrics.json').write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8'
)

print(f'\nartefactos en {OUTPUT_DIR}')
metrics['overall']

registros=2000 visibles=1873



artefactos en C:\Users\Cristian\Documents\UFC TRACKER\outputs\pose-merge\fiziev_bahamondes__rafael_fiziev_vs_ignacio_bahamondes__normal_men_round1


{'tracking_visible_frames': 1873,
 'pose_valid_frames': 1825,
 'pose_coverage': 0.974373,
 'mean_inference_ms_per_visible_frame': 14.0325}

In [9]:
for fighter_id, fighter_metrics in metrics['fighters'].items():
    print(
        f"{fighter_id}: visibles={fighter_metrics['tracking_visible_frames']} "
        f"coverage={fighter_metrics['pose_coverage']:.3f} "
        f"incompletos={fighter_metrics['incomplete_required_keypoint_frames']}"
    )
    availability = fighter_metrics['required_keypoint_availability']
    for name in ('left_wrist', 'right_wrist', 'left_ankle', 'right_ankle'):
        print(f"    {name:>12}: {availability[name]:.2f}")

fighter_left: visibles=989 coverage=0.954 incompletos=763
      left_wrist: 0.50
     right_wrist: 0.53
      left_ankle: 0.76
     right_ankle: 0.88
fighter_right: visibles=884 coverage=0.997 incompletos=547
      left_wrist: 0.87
     right_wrist: 0.61
      left_ankle: 0.96
     right_ankle: 0.95


## Criterios de revisión

Abre `PREVIEW_PATH` y verifica que **cada color siga al mismo peleador durante todo el round**, sobre todo en los frames donde antes cambiaba el `track_id` (cortes de cámara y clinch). Si el color salta de un cuerpo a otro, el merge unió fragmentos que no correspondían: baja `MAX_MERGE_DISTANCE` o acorta `MAX_MERGE_GAP_FRAMES`. Si en cambio un peleador desaparece por tramos largos, el merge quedó demasiado estricto y hay que aflojar esos mismos umbrales.

## Resumen del notebook (pose estimation + merge)

### Qué se hizo (visión general)

Variante del pipeline de pose que ataca el problema pendiente del notebook base: **ByteTrack parte a un mismo peleador en varios `track_id`**. En vez de exigirle al detector que entregue exactamente dos tracks, aquí los fragmentos se agrupan primero en dos identidades estables y después se estima la pose.

Flujo en capas:

1. **Tracking crudo** — `track_video()` sin filtro de rol; se conservan todos los fragmentos.
2. **Descripción y filtrado** — cada fragmento se resume (rango de frames, centroides, diagonal, piel, área) y se filtran los que parecen peleador.
3. **Merge codicioso** — se unen fragmentos disjuntos en el tiempo, cercanos en pantalla y con hueco corto; se conservan los dos slots más largos.
4. **Pose sobre identidades fusionadas** — `estimate_pose_records()` recibe `TrackingRecord` construidos a mano, con `fighter_left` / `fighter_right`.

---

### Funciones importantes

| Función | Rol |
|---------|-----|
| `describe_fragments()` | Resume cada `track_id`: rango temporal, centroides, diagonal, piel y área |
| `select_candidates()` | Descarta público y staff con los umbrales de piel/área heredados |
| `normalized_distance()` | Distancia entre centroides dividida por la diagonal de la caja |
| `merge_fragments()` | Agrupa fragmentos en slots por no solape + hueco corto + cercanía |
| `select_fighter_slots()` | Top-2 slots por frames, ordenados de izquierda a derecha |
| `build_tracking_records()` | Construye los `TrackingRecord`, incluyendo los frames no visibles |
| `estimate_pose_records()` | MediaPipe sobre cada crop (reutilizado del módulo de producción) |
| `render_pose_preview()` / `calculate_metrics()` | Preview y métricas, sin cambios respecto al pipeline base |

---

### Resultado sobre el round de prueba

`fiziev_bahamondes` round 1, primeros 1000 frames:

| | Sin merge | Con merge |
|---|---|---|
| Identidades finales | 3 (`2`, `61`, `807`) | 2 (`fighter_left`, `fighter_right`) |
| Frames con más de dos peleadores | 22 con el umbral por defecto | 0 |
| Frames con al menos un peleador | parcial | 1.000 |
| Filas visibles | 1835 | 1873 |
| `pose_coverage` global | 0.979 | 0.974 |

El merge reconstruyó `fighter_left = [2, 807]` (989 frames) y `fighter_right = [1, 61, 1553]` (884 frames). La cobertura de pose baja un poco porque ahora se procesan más frames, incluidos los tramos difíciles que antes quedaban fuera.

---

### Problemas encontrados

- **Fragmentación por cortes de cámara** — `tid=2` cubre los frames 0-589 y `tid=807` los 591-999; son el mismo peleador con 2 frames de hueco.
- **El umbral por frames no lo resuelve** — subir `min_track_frames` elimina fragmentos cortos, pero deja los tres fragmentos largos y el pipeline sigue viendo un tercer peleador.
- **Solape temporal como señal** — dos fragmentos que coinciden en un frame no pueden ser la misma persona; sin esta regla el merge unía a los dos peleadores.
- **Escala variable** — la distancia entre centroides en píxeles no sirve con zoom; hay que normalizarla por la diagonal de la caja.
- **Fragmentos de referee y público** — pasan el filtro de piel cuando hay poca ropa a la vista, pero quedan fuera al conservar solo los dos slots más largos.

---

### Métodos de solución importantes

1. **Tres condiciones simultáneas** — no solape temporal, hueco ≤ `MAX_MERGE_GAP_FRAMES` y distancia normalizada ≤ `MAX_MERGE_DISTANCE`.
2. **Invariante garantizada** — como dentro de un slot no hay solape, dos slots dan como máximo dos peleadores por frame; ya no hace falta abortar por conteo.
3. **Codicioso por orden de aparición** — cada fragmento se engancha al slot compatible más cercano, lo que evita decidir sobre todo el grafo de una vez.
4. **Top-2 en lugar de umbral** — se conservan los dos slots con más frames, no todos los que pasan un umbral.
5. **Trazabilidad** — el `track_id` original viaja en cada registro, así se puede auditar de qué fragmento salió cada caja.
6. **Eje temporal completo** — se emiten filas con `visible=False` para los frames sin caja.

---

### Parámetros clave (ajuste rápido)

| Parámetro | Uso |
|-----------|-----|
| `MAX_MERGE_GAP_FRAMES` | Cuánto puede desaparecer un peleador y seguir siendo el mismo (60 = 2 s) |
| `MAX_MERGE_DISTANCE` | Salto máximo en pantalla, en fracciones de la diagonal de la caja |
| `MIN_FRAGMENT_FRAMES` | Duración mínima para considerar un fragmento; bajo a propósito |
| `FIGHTER_SKIN_MIN` / `FIGHTER_MIN_AREA` | Filtro de apariencia heredado de person detection |
| `TRACKING_CONFIDENCE` | Umbral de detección antes de ByteTrack |
| `MAX_FRAMES` | Frames a analizar |

---

### Nota para el desarrollador — próxima mejora

- **Llevar el merge a producción** — mover `describe_fragments` / `merge_fragments` / `select_fighter_slots` a `ufc_tracker/tracking/`, y que `extract_fighter_tracking()` los use en vez de exigir exactamente dos tracks. Eso elimina el ajuste manual de `min_track_frames` en `poseEstimation.send_prediction()`.
- **Mapear a rojo / azul** — con identidades estables ya se puede asignar `fighter_red` / `fighter_blue` por color de short en vez de por posición.
- **Validar en varios rounds** — los umbrales están calibrados sobre un solo video; hay que revisarlos en clips con más cortes de cámara.
- **Re-ID por apariencia** — si el merge geométrico falla en peleas con mucho movimiento de cámara, el siguiente paso son embeddings (BoT-SORT, DeepSORT).
- **Registrar el pipeline en MLflow** — el merge cambia el resultado, así que merece ser una versión distinta del modelo.